In [1]:
import numpy as np
import sklearn as sk
import matplotlib
import matplotlib.pyplot as plt
import scipy.stats
from sklearn.linear_model import LogisticRegression
import pandas as pd

In [23]:
spam = pd.read_csv('ElemStatLearn_SpamData.csv')
spam.shape

(4601, 58)

In [3]:
spam.head()

,word_freq_make,word_freq_address,word_freq_all,word_freq_3d,word_freq_our,word_freq_over,word_freq_remove,word_freq_internet,word_freq_order,word_freq_mail,...,char_freq_semicol,char_freq_paren,char_freq_bracket,char_freq_exc,char_freq_dollar,char_freq_pound,capital_run_length_average,capital_run_length_longest,capital_run_length_total,spam
0,0.00,0.64,0.64,0.0,0.32,0.00,0.00,0.00,0.00,0.00,...,0.00,0.000,0.0,0.778,0.000,0.000,3.756,61,278,spam
1,0.21,0.28,0.50,0.0,0.14,0.28,0.21,0.07,0.00,0.94,...,0.00,0.132,0.0,0.372,0.180,0.048,5.114,101,1028,spam
2,0.06,0.00,0.71,0.0,1.23,0.19,0.19,0.12,0.64,0.25,...,0.01,0.143,0.0,0.276,0.184,0.010,9.821,485,2259,spam
3,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.00,0.137,0.0,0.137,0.000,0.000,3.537,40,191,spam
4,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.00,0.135,0.0,0.135,0.000,0.000,3.537,40,191,spam


In [4]:
X = spam.iloc[:,0:57]
y = np.where(spam['spam']=='spam',1,0)

In [11]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=2)

In [21]:
#The sklearn version of logistic regression automatically has a ridge penalty
#Large C means small penalty here (opposite of lambda)
lmfit = LogisticRegression(C=100000, max_iter=10000).fit(X_train, y_train)

yhat_lm = lmfit.predict(X_test)
np.mean(y_test!=yhat_lm)


0.07834101382488479

In [18]:
#The sklearn version of logistic regression automatically has a ridge penalty
#Large C means small penalty here (opposite of lambda)
lmfit = LogisticRegression(penalty='none',max_iter=10000).fit(X_train, y_train)

yhat_lm = lmfit.predict(X_test)
np.mean(y_test!=yhat_lm)


0.07702435813034891

In [10]:
from sklearn.linear_model import LogisticRegressionCV
lassofit = LogisticRegressionCV(penalty='l1', solver='liblinear', Cs=50).fit(X_train, y_train)
yhat_lasso = lassofit.predict(X_test)
np.mean(y_test!=yhat_lasso)


0.07570770243581304

In [25]:
lassofit.coef_

array([[-5.33422768e-01, -1.49776258e-01,  9.03806551e-02,
         1.25439481e+00,  3.64928926e-01,  6.09752615e-01,
         2.03254093e+00,  5.61124315e-01,  1.10195327e+00,
         5.04333488e-02, -3.57249262e-01, -1.10146346e-01,
        -4.95779868e-02,  7.67114296e-03,  2.62562406e+00,
         1.18031785e+00,  9.32426631e-01,  2.87729256e-02,
         1.09173263e-01,  8.03892923e-01,  1.99459624e-01,
         2.86677207e-01,  1.96193012e+00,  2.23906000e-01,
        -2.16183720e+00, -1.25524486e+00, -9.58876693e+00,
         3.11982967e-01, -2.05837174e+00, -1.93722368e+00,
         9.63220094e-01,  5.23868315e+00, -8.49666646e-01,
        -1.32078218e+01, -1.97628382e+00,  1.09940992e+00,
        -6.57051412e-01, -7.58953445e-01, -1.03205894e+00,
        -1.92864196e-01, -2.35209279e+01, -2.55811677e+00,
        -1.31386297e+00, -1.73046667e+00, -9.65649450e-01,
        -1.77948025e+00, -2.45932834e+00, -2.65423709e+00,
        -1.61546958e+00, -1.31269898e-01, -1.63935732e+0

In [26]:
(lassofit.coef_ == 0).sum()/len(lassofit.coef_[0])

0.0

In [ ]:
%%R -i X_train -i X_test -i y_train -i y_test

library(glmnet)
y_train = as.logical(y_train)
cvfit = cv.glmnet(as.matrix(X_train),y_train,family='binomial', type.measure='class')


In [ ]:
%%R

print(cvfit$cvm[which(cvfit$lambda==cvfit$lambda.1se)])
print(mean(as.logical(y_test)!= predict(cvfit,newx=as.matrix(X_test),s=cvfit$lambda.1se,type='class')))
print(coef(cvfit, s=cvfit$lambda.1se))

In [ ]:
%%R
coefs = coef(cvfit, s=cvfit$lambda.1se)
rownames(coefs)[order(abs(coefs), decreasing=TRUE)]